# Edge S2 feature pass (embeddings, hashes, SQ-GAN, decode, timings) + SPI grid

Step `edge-s2`. Thin wrapper; logic lives in `Codec.edge.feature_pass`.


In [ ]:
import os, time, warnings
os.environ.update(PYTHONWARNINGS='ignore', PYDEVD_DISABLE_FILE_VALIDATION='1', TOKENIZERS_PARALLELISM='false',
                  HF_HUB_DISABLE_PROGRESS_BARS='1', TQDM_DISABLE='1')
warnings.filterwarnings('ignore')
T0 = time.time()

In [ ]:
import os, subprocess
REPO, URL, BRANCH = "/kaggle/working/repo", "https://github.com/saltypal/SemanticSchedulerEnd2End.git", "main"
def _git(*a, cwd=None):
    return subprocess.run(["git", *a], cwd=cwd, check=True, capture_output=True, text=True).stdout.strip()
if os.path.isdir(REPO + "/.git"):
    _git("fetch", "-q", "--depth", "1", "origin", BRANCH, cwd=REPO)
    _git("reset", "-q", "--hard", "FETCH_HEAD", cwd=REPO)
else:
    _git("clone", "--depth", "1", "-q", "--no-progress", "--branch", BRANCH, URL, REPO)
GIT_SHA = _git("rev-parse", "HEAD", cwd=REPO)
print("repo @", GIT_SHA[:8], "(" + _git("log", "-1", "--format=%s", cwd=REPO) + ")")

In [ ]:
import subprocess, time
_t = time.time()
LOG = "/kaggle/working/pip_install.log"
with open(LOG, "w") as _f:
    _rc = subprocess.run(["pip", "install", "-q", "--no-input", "--disable-pip-version-check",
                          "--ignore-requires-python", "-e", ".", *['lpips']],
                         cwd=REPO, stdout=_f, stderr=subprocess.STDOUT).returncode
if _rc:
    print("".join(open(LOG).readlines()[-25:]))
    raise RuntimeError("pip install failed (rc=%s)" % _rc)
print("installed in %.0fs" % (time.time() - _t))

In [ ]:
import os, subprocess
ARGV = ['python', '-u', '-m', 'Codec.edge.feature_pass', 'run', '--out', '/kaggle/working/runs/edge-s2']
print("$ python -m Codec.edge.feature_pass " + " ".join(ARGV[4:]))
_p = subprocess.Popen(ARGV, cwd=REPO, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end="", flush=True)
if _p.wait():
    raise RuntimeError("Codec.edge.feature_pass exited with %s" % _p.returncode)

## Compute profile (once per model)


In [ ]:
import os, subprocess
ARGV = ['python', '-u', '-m', 'Codec.edge.compute_profile', '--out', '/kaggle/working/runs/edge-s2']
print("$ python -m Codec.edge.compute_profile " + " ".join(ARGV[4:]))
_p = subprocess.Popen(ARGV, cwd=REPO, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end="", flush=True)
if _p.wait():
    raise RuntimeError("Codec.edge.compute_profile exited with %s" % _p.returncode)

## Grid hand-off

Provisional splits are built by `parameters.edge.splits.assign_event_splits` when `splits_v1.json` is absent from the labels dataset; the export step prints which was used.


In [ ]:
import os, subprocess
ARGV = ['python', '-u', '-m', 'Codec.edge.feature_pass', 'export-grid', '--out', '/kaggle/working/runs/edge-s2']
print("$ python -m Codec.edge.feature_pass " + " ".join(ARGV[4:]))
_p = subprocess.Popen(ARGV, cwd=REPO, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end="", flush=True)
if _p.wait():
    raise RuntimeError("Codec.edge.feature_pass exited with %s" % _p.returncode)

In [ ]:
import os, subprocess
ARGV = ['python', '-u', '-m', 'parameters.edge.embedding_grid', 'grid', '--embeddings-dir', '/kaggle/working/runs/edge-s2/grid', '--labels', '/kaggle/working/runs/edge-s2/grid/labels_by_uid.jsonl', '--splits', '/kaggle/working/runs/edge-s2/grid/splits_by_uid.json', '--out', '/kaggle/working/runs/edge-s2/grid']
print("$ python -m parameters.edge.embedding_grid " + " ".join(ARGV[4:]))
_p = subprocess.Popen(ARGV, cwd=REPO, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end="", flush=True)
if _p.wait():
    raise RuntimeError("parameters.edge.embedding_grid exited with %s" % _p.returncode)

In [ ]:
import os, subprocess
ARGV = ['python', '-u', '-m', 'parameters.edge.embedding_grid', 'select', '--grid-dir', '/kaggle/working/runs/edge-s2/grid']
print("$ python -m parameters.edge.embedding_grid " + " ".join(ARGV[4:]))
_p = subprocess.Popen(ARGV, cwd=REPO, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end="", flush=True)
if _p.wait():
    raise RuntimeError("parameters.edge.embedding_grid exited with %s" % _p.returncode)

In [ ]:
import datetime, json, os, platform, subprocess, time
STEP, ARGS = "edge-s2", ['run', '--out', '/kaggle/working/runs/edge-s2']
def _gpu():
    try:
        return subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    except Exception:
        return None
try:
    import torch; torch_version = torch.__version__
except Exception:
    torch_version = None
out = "/kaggle/working/runs/" + STEP
os.makedirs(out, exist_ok=True)
manifest = {"step": STEP, "entrypoint": "Codec.edge.feature_pass", "args": ARGS, "git_sha": GIT_SHA,
            "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "python": platform.python_version(),
            "torch": torch_version, "gpu": _gpu(), "wall_time_s": time.time() - T0}
json.dump(manifest, open(out + "/run_manifest.json", "w"), indent=2)
print(manifest)

## Publish cache (dataset scheduler-dataset-v1-cache)

Skipped with a reason if no credentials.


In [ ]:
import os, subprocess
ARGV = ['python', '-u', '-m', 'Codec.edge.feature_pass', 'publish', '--out', '/kaggle/working/runs/edge-s2', '--slug', 'scheduler-dataset-v1-cache']
print("$ python -m Codec.edge.feature_pass " + " ".join(ARGV[4:]))
_p = subprocess.Popen(ARGV, cwd=REPO, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end="", flush=True)
if _p.wait():
    raise RuntimeError("Codec.edge.feature_pass exited with %s" % _p.returncode)